In [10]:
import numpy as np
from numpy import random
import matplotlib.pyplot as plt
%matplotlib inline

In [11]:
def propensity(k,*args):
    
    for i in args:
        k=k*i
    return k


def reaction1(a,b,c):
    '''the binding of ribosome into RNA'''
    
    if a>0 and b>0:
        a-=1
        b-=1
        c+=1
    return a,b,c


def reaction2(a,b,c):
    '''Ribosome read through the second codon, release the RBS of mRNA'''
    
    if a>0:
        a-=1
        b+=1
        c+=1
    return a,b,c


def reaction3(a,b):
    '''ribosome read through mRNA, actually the transition of complex
    or encounter the stop codon, release the free ribsome'''
    
    if a>0:
        a-=1
        b+=1
    return a,b


def translation(sequence):  # how many codons in the sequence
    
    codon_dict={'UUU':'F', 'UUC':'F', 'UUA':'L', 'UUG':'L',     
                 'UCU':'S', 'UCC':'S', 'UCA':'S', 'UCG':'S',     
                 'UAU':'Y', 'UAC':'Y', 'UAA':'STOP', 'UAG':'STOP',     
                 'UGU':'C', 'UGC':'C', 'UGA':'STOP', 'UGG':'W',     
                 'CUU':'L', 'CUC':'L', 'CUA':'L', 'CUG':'L',     
                 'CCU':'P', 'CCC':'P', 'CCA':'P', 'CCG':'P',     
                 'CAU':'H', 'CAC':'H', 'CAA':'Q', 'CAG':'Q',     
                 'CGU':'R', 'CGC':'R', 'CGA':'R', 'CGG':'R',     
                 'AUU':'I', 'AUC':'I', 'AUA':'I', 'AUG':'M',     
                 'ACU':'T', 'ACC':'T', 'ACA':'T', 'ACG':'T',     
                 'AAU':'N', 'AAC':'N', 'AAA':'K', 'AAG':'K',     
                 'AGU':'S', 'AGC':'S', 'AGA':'R', 'AGG':'R',     
                 'GUU':'V', 'GUC':'V', 'GUA':'V', 'GUG':'V',     
                 'GCU':'A', 'GCC':'A', 'GCA':'A', 'GCG':'A',     
                 'GAU':'D', 'GAC':'D', 'GAA':'E', 'GAG':'E',    
                 'GGU':'G', 'GGC':'G', 'GGA':'G', 'GGG':'G'}
    
    aa_list=[]
    
    start_point=sequence.find('AUG')
    trimmed_sequence=sequence[start_point:]
    
    codons_list = [trimmed_sequence[i:i+3] for i in range(0, len(trimmed_sequence), 3)]
    
    stop_list=['UAA', 'UAG','UGA']
    
    n=0
    for i in codons_list:
        if len(i)==1 or len(i)==2:
            pass
        elif i not in stop_list:
            n+=1
            aa_list.append(codon_dict[i])
        else:
            break
            
    return n,aa_list
    

def directmethod(sequence,initialcomp_list,reconstant_list, total_rib,rna, t1, steps): 
    
    complex_list=initialcomp_list
    rib=total_rib
    rna_rbs=rna
    
    s=translation(sequence)
    n=s[0]
    aa_list=s[1]
    
    prop_list=[]
    prop1=propensity(reconstant_list[0],rib,rna_rbs)
    
    prop_list.append(prop1)
         
    for i in range(1,n):
        prop_list.append(propensity(reconstant_list[i],complex_list[i-1]))
    
    if steps:
        new_prop = []
        for i, p in enumerate(prop_list):
            if i not in steps:
                new_prop.append(p)
            else:
                new_prop.append(0)
    else:
        new_prop = prop_list
        
        
    tau_list=[]
    
    for i in new_prop:
        if i==0:
            tau_list.append(np.inf)
        else:
            tau_list.append(((1/i)*np.log(1/(1-np.random.random()))))
    
 
    if sum(new_prop)!=0:
        tau=min(tau_list)
        u=np.argmin(tau_list)
                
        if u==0:
#             print('Reaction start', t1)
            rib,rna_rbs,complex_list[0]=reaction1(rib,rna_rbs,complex_list[0])  
#             print(rib,rna_rbs,complex_list[0])
        elif u==1:
#             print('First position',t1)
            complex_list[0],complex_list[1],rna_rbs=reaction2(complex_list[0],complex_list[1],rna_rbs) # release the RBS
        elif u==n-1:
#             print('Stop',t1)
            complex_list[u-1],rib=reaction3(complex_list[u-1],rib) # encouter the STOP codon
        else:
#             print('Elongation', u,t1)
            complex_list[u-1],complex_list[u]=reaction3(complex_list[u-1],complex_list[u])
        
    newcomplex_list=[]  
    for i in range(len(complex_list)):
        newcomplex_list.append(complex_list[i])
        
    rib1=rib
    rna_rbs1=rna_rbs
    
    aa=aa_list[u]
                
    return newcomplex_list, rib1, rna_rbs1, tau, aa, u
        
        

def simulation(sequence,initialcomp_list,reconstant_list,total_rib,rna,t1,t2):
    
    t0=t1
    t1_list=[]
    
    complex_list=initialcomp_list
    chemical_list=[]  # will be used to store the complex_list of each step
    rib=total_rib
    
    n=translation(sequence)[0]
    
    rna_rbs=rna
    
    result={}
    aa_result={}
    
    ids=0
    u_list=[]
    
    steps=[]

    while t1<t2:
#         print(t1)
        y=directmethod(sequence,complex_list,reconstant_list, rib,rna_rbs, t1,steps)
        u=y[-1]
        aa=y[-2]
        print('U is', u)
        
        if u_list==[]:
            complex_list=y[0]
            rib=y[1]
            rna_rbs=y[2]

            t1+=y[3]
            t1_list.append(t1)
            
            u_list.append(u)
            ids+=1
            result[ids]=[(t1,u)]
            aa_result[ids]=[aa]
            
            steps=[u]
            
            print('Initial Steps',steps)
        
        else:

            complex_list=y[0]
            rib=y[1]
            rna_rbs=y[2]

            u_list.append(u)
                
            if u==0:
                ids+=1
                t1 += y[3]
                result[ids]=[(t1,u)]
                aa_result[ids]=[aa]
            else:
                t1 += y[3]
                for k, v in result.items():
                    if v[-1][-1]==u-1:
                        result[k].append((t1, u))
                        aa_result[k].append(aa)
                        break

            steps = [v[-1][-1] for v in result.values() if v[-1][-1]!=n-1]
            steps = [v - 1 for v in steps if v!=0] + steps    
            
            print('STEPS', steps)
    
    for k,v in aa_result.items():
#         print('Reaction {}:'.format(k),':' ,''.join(v))
        print('Reaction {}: {}'.format(k,''.join(v)))        
    
    return result,aa_result
       


In [12]:
#sequence='ACAUGCUAGAACCGCAUGUACUAGUUAA'
sequence='AUGCUAGAACCGCUAGAACCGCAUGUACAAGUUAACAAGAACCGCUAGAACCGCAUGUACUAGUUAA'

s=translation(sequence)
n=s[0]

initialcomp_list=[]
for i in range(0,n-1):
    initialcomp_list.append(0)

# print(initialcomp_list)

reconstant_list=[1]
for i in range(n-1):
    reconstant_list.append(2)
# print(reconstant_list)


total_rib=5
rna=1

t1=1
t2=100

# x=directmethod(sequence,initialcomp_list,reconstant_list,total_rib,rna)
# print(x[0])
# print(x[1])
# print(x[2])
# print(x[3])
# print(x[4])
# print(x[5])

x=simulation(sequence,initialcomp_list,reconstant_list,total_rib,rna,t1,t2)
# for i in zip(x, y):
#     print(i)
print(x)

U is 0
Initial Steps [0]
U is 1
STEPS [0, 1]
U is 2
STEPS [1, 2]
U is 3
STEPS [2, 3]
U is 0
STEPS [2, 3, 0]
U is 1
STEPS [2, 0, 3, 1]
U is 4
STEPS [3, 0, 4, 1]
U is 2
STEPS [3, 1, 4, 2]
U is 0
STEPS [3, 1, 4, 2, 0]
U is 5
STEPS [4, 1, 5, 2, 0]
U is 6
STEPS [5, 1, 6, 2, 0]
U is 7
STEPS [6, 1, 7, 2, 0]
U is 3
STEPS [6, 2, 7, 3, 0]
U is 1
STEPS [6, 2, 0, 7, 3, 1]
U is 8
STEPS [7, 2, 0, 8, 3, 1]
U is 4
STEPS [7, 3, 0, 8, 4, 1]
U is 9
STEPS [8, 3, 0, 9, 4, 1]
U is 5
STEPS [8, 4, 0, 9, 5, 1]
U is 2
STEPS [8, 4, 1, 9, 5, 2]
U is 10
STEPS [9, 4, 1, 10, 5, 2]
U is 11
STEPS [10, 4, 1, 11, 5, 2]
U is 0
STEPS [10, 4, 1, 11, 5, 2, 0]
U is 6
STEPS [10, 5, 1, 11, 6, 2, 0]
U is 7
STEPS [10, 6, 1, 11, 7, 2, 0]
U is 8
STEPS [10, 7, 1, 11, 8, 2, 0]
U is 12
STEPS [11, 7, 1, 12, 8, 2, 0]
U is 9
STEPS [11, 8, 1, 12, 9, 2, 0]
U is 10
STEPS [11, 9, 1, 12, 10, 2, 0]
U is 3
STEPS [11, 9, 2, 12, 10, 3, 0]
U is 13
STEPS [12, 9, 2, 13, 10, 3, 0]
U is 1
STEPS [12, 9, 2, 0, 13, 10, 3, 1]
U is 14
STEPS [9, 2, 0, 10, 